In [ ]:
perm_code_avoid_map = dict()

# returns (perm, codes)
def get_perm_code_avoid(n, pt):
    key = (n, '.'.join([str(num) for num in pt]))
    if not key in perm_code_avoid_map:
        perm = []
        code = []
        for p in Permutations(n):
            if p.avoids(pt):
                perm.append(p)
                code.append(p.to_lehmer_code())

        perm_avoid_map[key] = (perm, code)

    return perm_avoid_map[key]

perm_avoid_map = dict()

# 
def get_perm_avoid(n, pt):
    key = (n, '.'.join([str(num) for num in pt]))

    if not key in perm_avoid_map:
        perm_list = []
 
        for p in Permutations(n):
            if p.avoids(pt):
                perm_list.append(p)

        perm_avoid_map[key] = perm_list

    return perm_avoid_map[key]

    

In [ ]:
a123 = [1,2,3]
a132 = [1,3,2]
a213 = [2,1,3]
a231 = [2,3,1]
a312 = [3,1,2]
a321 = [3,2,1]

def findavoid(n,pt, d='c'):
    pattern, code = get_perm_code_avoid(n, pt)
    
    if d=='pa':
        return pattern
    if d=='c':
        return code
    if d=='l':
        return len(pattern)

def findpat(n,pt,d):
    pattern = []
    code = []
    for p in Permutations(n):
        if p.has_pattern([1,2,3]):
            pattern.append(p)
            code.append(p.to_lehmer_code())
    if d=="pa":
        return pattern
    if d=='c':
        return code
    if d=='l':
        return len(pattern)

In [2]:
#pick out all the one-return dyck path(lehmer code) by locating all the two+ return ones
#findflush is to find all n lehmer code that have "flush" property: "flush" property means exist
#1.top digit (reach the possible maximam) except for the 0 at last. (eg. in [4,3,4,2,1,1,0], the 2nd "4", the last "1" are top digits)
#2.all the digits before this digit are larger or equal to this digit. (eg. 4,3,4,2,1 >= 1, so [4,3,4,2,1,1,0] flush).
def findflush(n, pt):
    flushcode = []
    perm_avoid_list = get_perm_avoid(n, pt)

    for p in perm_avoid_list:
        c = p.to_lehmer_code()
        for i in range(0, n - 1):
            val = c[i]
            if val == n - i - 1:
                if all(c[j] >= val for j in range(i)):
                    flushcode.append(c)
                    break
    return flushcode

#this function find all lehmer code without flush property because we know flush lehmer code biject with 2+ return dyck path
def findonereturn(n,pt):
    flush_code_list = findflush(n,pt)
    code_list = []
    perm_avoid_list = get_perm_avoid(n, pt)
    for p in perm_avoid_list:
        c = p.to_lehmer_code()
        if c not in flush_code_list:
            code_list.append(c)
    return code_list

#from n-1 lehmer code, I add the "largest not flush" number to the 2nd position, then form a n lehmer code
def is_flush(c):
    n = len(c)
    for i in range(n - 1):
        val = c[i]
        if val == n - i - 1:
            if all(c[j] >= val for j in range(i)):
                return True
    return False

def insert_max_nonflush_digit(p):
    n = len(p) + 1
    for x in reversed(range(n - 1)):
        new_code = [p[0], x] + p[1:]
        if not is_flush(new_code):
            return new_code
    return None 

In [3]:
def custom_sort(lst):
    return sorted(lst, key=lambda x: (x[0], x[2], x[3], x[4], x[5],x[6],x[1]))

In [4]:
#this is for rearranging. The rule is: I fix all "top" digits, then I rearrange all non-top digits in a decreasing order.
def rearrange_nonflush_descending(p):
    n = len(p)
    fixed_indices = set()
    #fixed_indices.add(0) I found no need, it's always the largest (haven't prove)
    for i in range(n):
        if p[i] == n - i - 1:
            fixed_indices.add(i)
    to_sort = [p[i] for i in range(n) if i not in fixed_indices]
    to_sort_sorted = sorted(to_sort, reverse=True)
    pnew = []
    j = 0
    for i in range(n):
        if i in fixed_indices:
            pnew.append(p[i])
        else:
            pnew.append(to_sort_sorted[j])
            j += 1
    return pnew

In [6]:
#this is for undo the process. 
# The rule is: I fix all "top" digits, 
# then move smallest digit to front.
def undo_nonflush_descending(p):
    n = len(p)
    fixed_indices = set()
    #fixed_indices.add(0) I found no need, it's always the largest (haven't prove)
    for i in range(n):
        if p[i] == n - i - 1:
            fixed_indices.add(i)
    to_undo = [p[i] for i in range(n) if i not in fixed_indices]
    undo_list = []
    # for all not top (except index 0)
    for j in range(1,len(to_undo)):
        temp = [x for x in to_undo]
        # what if x was the inserted value
        x = temp.pop(j)
        temp.insert(1, x)
        pnew = []
        j = 0
        for i in range(n):
            if i in fixed_indices:
                pnew.append(p[i])
            else:
                pnew.append(temp[j])
                j += 1
        undo_list.append(pnew)

    ret_val = None

    for undo in undo_list:
        undo2 = [x for x in undo]
        undo2.pop(1)
        if not is_flush(undo) and is_flush(undo2):
            temp_undo = [x for x in undo]
            temp_undo[1] = temp_undo[1] + 1
            if is_flush(temp_undo):
                #print('not', undo, 'yes', undo2, 'yes', temp_undo)
                retval = undo
                break
    return undo

In [7]:
undo_nonflush_descending([5, 2, 4, 0, 2, 1, 0])

[5, 0, 4, 2, 2, 1, 0]

In [271]:
oldlist = findonereturn(7,a123) #takes very long time, input n

In [272]:
len(oldlist)

132

In [274]:
newlist = []
for p in findavoid(6, a123, "c"): #from n-1 lehmer code, then we "add leg", input n-1
    newlist.append(insert_max_nonflush_digit(p))

In [275]:
print(len(newlist))

132


In [276]:
#rearrange, but I think for most of them, they have the required arrangement already: probobaly it's the condition of 123-avoding.
for i, p in enumerate(newlist):
   pnew = rearrange_nonflush_descending(p)
   if not p == pnew:
       #print(p, pnew)
       pold = [x for x in p]
       pold.pop(1)
       pnew_undo = undo_nonflush_descending(pnew)
       pnew_undo.pop(1)
       if not pold == pnew_undo:
           print('\tdidnt work', pold, pnew_undo)
   newlist[i] = pnew
print('done')

done


In [278]:
list1 = custom_sort(oldlist)
list2 = custom_sort(newlist)

In [279]:
#test difference
diff1 = [item for item in list1 if item not in list2]
diff2 = [item for item in list2 if item not in list1]
print(diff1)
print(diff2)
newlist == oldlist

[]
[]


False

In [280]:
list1 == list2


True

In [59]:
for p in Permutations(3):
    print(p.to_lehmer_code())

[0, 0, 0]
[0, 1, 0]
[1, 0, 0]
[1, 1, 0]
[2, 0, 0]
[2, 1, 0]


In [82]:
pa_list = get_perm_avoid(5,a123)
for p in pa_list:
    print(p.to_lehmer_code())

returning [[1, 5, 4, 3, 2], [2, 1, 5, 4, 3], [2, 5, 1, 4, 3], [2, 5, 4, 1, 3], [2, 5, 4, 3, 1], [3, 1, 5, 4, 2], [3, 2, 1, 5, 4], [3, 2, 5, 1, 4], [3, 2, 5, 4, 1], [3, 5, 1, 4, 2], [3, 5, 2, 1, 4], [3, 5, 2, 4, 1], [3, 5, 4, 1, 2], [3, 5, 4, 2, 1], [4, 1, 5, 3, 2], [4, 2, 1, 5, 3], [4, 2, 5, 1, 3], [4, 2, 5, 3, 1], [4, 3, 1, 5, 2], [4, 3, 2, 1, 5], [4, 3, 2, 5, 1], [4, 3, 5, 1, 2], [4, 3, 5, 2, 1], [4, 5, 1, 3, 2], [4, 5, 2, 1, 3], [4, 5, 2, 3, 1], [4, 5, 3, 1, 2], [4, 5, 3, 2, 1], [5, 1, 4, 3, 2], [5, 2, 1, 4, 3], [5, 2, 4, 1, 3], [5, 2, 4, 3, 1], [5, 3, 1, 4, 2], [5, 3, 2, 1, 4], [5, 3, 2, 4, 1], [5, 3, 4, 1, 2], [5, 3, 4, 2, 1], [5, 4, 1, 3, 2], [5, 4, 2, 1, 3], [5, 4, 2, 3, 1], [5, 4, 3, 1, 2], [5, 4, 3, 2, 1]]
[0, 3, 2, 1, 0]
[1, 0, 2, 1, 0]
[1, 3, 0, 1, 0]
[1, 3, 2, 0, 0]
[1, 3, 2, 1, 0]
[2, 0, 2, 1, 0]
[2, 1, 0, 1, 0]
[2, 1, 2, 0, 0]
[2, 1, 2, 1, 0]
[2, 3, 0, 1, 0]
[2, 3, 1, 0, 0]
[2, 3, 1, 1, 0]
[2, 3, 2, 0, 0]
[2, 3, 2, 1, 0]
[3, 0, 2, 1, 0]
[3, 1, 0, 1, 0]
[3, 1, 2, 0, 0]
[3,

In [103]:
rearrange_nonflush_descending( [5, 1, 4, 2, 2, 0, 0])

[5, 2, 4, 1, 2, 0, 0]

In [104]:
rearrange_nonflush_descending( [5, 0, 4, 2, 2, 1, 0])

[5, 2, 4, 0, 2, 1, 0]

In [184]:
undo_nonflush_descending([5, 2, 4, 0, 2, 1, 0])

[[5, 0, 4, 2, 1, 0], [5, 2, 4, 2, 1, 0]]

In [115]:
rearrange_nonflush_descending( [5, 0, 4, 2, 2, 1, 0])

[5, 2, 4, 0, 2, 1, 0]